# Does a ridge help mask-only registration beyond searching harder? All P2I-LReg patients

Design, conditions and the decision rule: the docstring of `registration/test_ridge_value.py` (written below).
Three conditions per view: `outline` (mask only), `ridge` (mask + a perfect ridge), `outline_more`
(mask only, 4x the search). Primary result: ridge - outline_more on the 17 patients not used in development.

**Before running**
1. *Add Input*: the dataset `junezhou001/p2i-lreg`.
2. *Settings*: **Internet on** (for `pip install open3d`); accelerator **None** (CPU only; a GPU does not help).
3. *Save Version -> Save & Run All*. The run stops itself after `HOURS` and keeps every finished view.
   If the summary says views are missing, add this version's output as an input (*Add Input* -> *Notebook Output*)
   and run a new version: finished views are copied in and not recomputed.

In [ ]:
HOURS = 11                 # Kaggle stops a run at 12 h; stop ours earlier so the output is saved
VIEWS_PER_PATIENT = 10     # fixed in test_ridge_value.py's docstring

## 1. Environment

In [ ]:
!pip install -q open3d
import os, sys, numpy, scipy, open3d
WORKERS = os.cpu_count()
print(sys.version.split()[0], "numpy", numpy.__version__, "scipy", scipy.__version__,
      "open3d", open3d.__version__, "cpus", WORKERS)

## 2. Find the dataset and the patients

In [ ]:
import glob
# e.g. /kaggle/input/datasets/junezhou001/p2i-lreg/Liver_regis/01/model/reconstructed_mesh_world_m.obj
hits = sorted(set(os.path.dirname(os.path.dirname(os.path.dirname(p)))
                  for depth in range(1, 7)
                  for p in glob.glob("/kaggle/input/" + "*/" * depth + "model/reconstructed_mesh_world_m.obj")))
assert len(hits) == 1, f"expected one dataset root, found {hits}"
ROOT = hits[0]
PATIENTS = sorted(p for p in os.listdir(ROOT) if p.isdigit() and os.path.exists(f"{ROOT}/{p}/syn/camPose.yml"))
DEV = ["01", "06", "10", "11"]
ORDER = [p for p in PATIENTS if p not in DEV] + [p for p in PATIENTS if p in DEV]  # confirmatory patients first
print("dataset root:", ROOT)
print(len(PATIENTS), "patients, run in this order:", " ".join(ORDER))

## 3. Our scripts (embedded from the repo by make_notebook.py)

In [ ]:
os.makedirs('/kaggle/working/benchmark', exist_ok=True); os.makedirs('/kaggle/working/registration', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/benchmark/pilot_dice_vs_tumour.py
"""
Pilot: does a high Dice guarantee that the tumour is in the right place?

On synthetic P2I-LReg views (true camera pose known) we plant spherical tumours inside
the CT liver, produce wrong pose predictions, and score every prediction two ways:
  - Dice between the projected CT outline and the liver label (what papers report)
  - tumour error: mm between where the prediction puts each tumour and where it really is
A one-sided surface-fit score (visible cloud -> CT surface, mm) is reported as well,
since that is the other score that can be computed without ground truth.

Wrong predictions come from five families, all sampled the same way:
  camera   rotation about the camera centre + translation  (error in camera orientation)
  visible  rotation about the centroid of the visible surface + translation
  organ    rotation about the centroid of the whole liver + translation
  scale    zoom about the camera centre only (Dice is unchanged by construction; reported, not used)
  icp      point-to-plane ICP of the visible cloud to the CT, started from an 'organ' error.
           These are outputs of a real algorithm, so they are the primary test.
Rotation angle ~ U(0, 15 deg) about a uniformly random axis; translation ~ U(0, 20 mm)
in a uniformly random direction; scale ~ log-U(0.8, 1.25). Views are drawn at random from
each patient's test split, and tumours uniformly from the whole liver volume (not from the
visible part). All ranges and the decision rule below were fixed before the first run.

Pre-registered decision rule (Dice >= 0.90, mean tumour error over all tumours):
  SUPPORTED      >= 10 % of predictions have tumour error > 10 mm
  NOT SUPPORTED  <  2 % have tumour error > 10 mm AND Spearman |rho(Dice, error)| >= 0.8
  INCONCLUSIVE   otherwise

Amendment after a 6-view smoke test, before the full run: on the clean synthetic cloud, ICP
converged to < 0.5 mm from every start, so the 'icp' family cannot produce realistic failures.
Added 'icp_scaled': the same ICP run on the cloud scaled about the camera centre by
s ~ log-U(1.01, 1.32), the 10-90 % range of the PCA scale-recipe error we measured on 238
synthetic views. The decision rule is applied to 'icp_scaled'; 'icp' is still reported.
Simplifications: rigid only (synthetic livers are not deformed), spheres with guessed radius.

Usage: python benchmark/pilot_dice_vs_tumour.py [--patients 01 06 10 11] [--views 25] ...
"""
import argparse
import csv
import os

import cv2
import numpy as np
import open3d as o3d
import yaml

DATA_DIR = "dataset/liver-registration"
H, W = 540, 960                      # synthetic render size
LIVER, TIP, SHAFT = 0, 153, 179      # synthetic label values (102 = background)

MAX_ROT_DEG = 15.0
MAX_TRANS_MM = 20.0
SCALE_RANGE = (0.8, 1.25)
SCALE_RECIPE_ERROR = (1.01, 1.32)    # 10-90 % range of the PCA scale-recipe error on synthetic views
TUMOUR_RADIUS_MM = (5.0, 25.0)       # 1-5 cm diameter; to be replaced by LiTS statistics
DICE_GOOD = 0.90
MARGIN_MM = 10.0
FAMILIES = ["camera", "visible", "organ", "scale", "icp", "icp_scaled"]
PRIMARY = "icp_scaled"


# ==============================================================================
# Loading
# ==============================================================================
def load_patient(pid: str):
    """CT mesh in mm, ray-casting scene, ICP target, K, poses, test views."""
    root = os.path.join(DATA_DIR, pid)
    mesh = o3d.io.read_triangle_mesh(os.path.join(root, "model", "reconstructed_mesh_world_m.obj"))
    mesh.scale(1000.0, center=[0, 0, 0])
    mesh.compute_vertex_normals()

    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))

    target = mesh.sample_points_uniformly(200000, use_triangle_normal=True).voxel_down_sample(2.0)

    with open(os.path.join(root, "syn", "CAM_K.yml")) as f:
        K = np.array(yaml.safe_load(f)["00000"][0]["cam_K"]).reshape(3, 3)
    v, u = np.mgrid[0:H, 0:W]
    pix_dirs = np.stack([(u + 0.5 - K[0, 2]) / K[0, 0], (v + 0.5 - K[1, 2]) / K[1, 1],
                         np.ones((H, W))], axis=-1).reshape(-1, 3)  # one ray per pixel centre
    with open(os.path.join(root, "syn", "camPose.yml")) as f:
        poses = yaml.safe_load(f)
    with open(os.path.join(root, "test_syn.txt")) as f:
        views = [int(line.strip().replace("ptsLiverCam", "")) for line in f if line.strip()]

    return dict(root=root, mesh=mesh, centroid=np.asarray(mesh.vertices).mean(axis=0), scene=scene,
                target=target, K=K, pix_dirs=pix_dirs, poses=poses, views=views)


def true_w2c(pose) -> np.ndarray:
    """4x4 world(mm) -> OpenCV camera(mm), from Blender camera-to-world (-Z forward)."""
    R_cv = np.array(pose["cam_R_c2w"]).reshape(3, 3) @ np.diag([1, -1, -1])
    t = np.array(pose["cam_t_c2w"]).reshape(3)
    T = np.eye(4)
    T[:3, :3] = R_cv.T
    T[:3, 3] = -R_cv.T @ t
    return T


def load_view(P, idx: int):
    label = cv2.imread(os.path.join(P["root"], "syn", "labels", f"lbl{idx:05d}.png"))[..., 0]
    pcd = o3d.io.read_point_cloud(os.path.join(P["root"], "syn", "liverPcds", f"ptsLiverCam{idx:05d}.ply"))
    pcd = pcd.voxel_down_sample(3.0)
    pts = np.asarray(pcd.points) * np.array([1.0, -1.0, -1.0])  # Blender camera -> OpenCV camera
    return label, pts


# ==============================================================================
# Geometry helpers
# ==============================================================================
def apply(T: np.ndarray, X: np.ndarray) -> np.ndarray:
    return X @ T[:3, :3].T + T[:3, 3]


def project(X: np.ndarray, K: np.ndarray) -> np.ndarray:
    z = np.where(X[:, 2] > 1e-6, X[:, 2], np.nan)
    return np.stack([K[0, 0] * X[:, 0] / z + K[0, 2], K[1, 1] * X[:, 1] / z + K[1, 2]], axis=1)


def render_mask(T_pred: np.ndarray, P) -> np.ndarray:
    """Liver outline seen with pose T_pred: cast one ray per pixel into the CT mesh."""
    inv = np.linalg.inv(T_pred)
    dirs = P["pix_dirs"] @ inv[:3, :3].T
    rays = np.concatenate([np.broadcast_to(inv[:3, 3], dirs.shape), dirs], axis=1).astype(np.float32)
    t_hit = P["scene"].cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()
    return np.isfinite(t_hit).reshape(H, W)


def dice(pred: np.ndarray, liver: np.ndarray, ignore: np.ndarray) -> float:
    """Dice over pixels not covered by a tool (the liver behind a tool is unknown)."""
    p, g = pred & ~ignore, liver & ~ignore
    return 2.0 * (p & g).sum() / max(p.sum() + g.sum(), 1)


def random_rotation(rng, max_deg: float) -> np.ndarray:
    axis = rng.normal(size=3)
    axis /= np.linalg.norm(axis)
    a = np.deg2rad(rng.uniform(0, max_deg))
    Kx = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(a) * Kx + (1 - np.cos(a)) * Kx @ Kx


def rigid_error(rng, pivot: np.ndarray) -> np.ndarray:
    """4x4 camera-frame error: rotate about the pivot, then translate."""
    R = random_rotation(rng, MAX_ROT_DEG)
    d = rng.normal(size=3)
    t = d / np.linalg.norm(d) * rng.uniform(0, MAX_TRANS_MM)
    E = np.eye(4)
    E[:3, :3] = R
    E[:3, 3] = pivot - R @ pivot + t
    return E


def rotation_deg(R: np.ndarray) -> float:
    return np.rad2deg(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))


def sample_tumours(P, rng, n: int):
    """Sphere centres uniform in the liver volume, each sphere fully inside the liver."""
    lo, hi = P["mesh"].get_min_bound(), P["mesh"].get_max_bound()
    centres, radii = [], []
    while len(centres) < n:
        c = rng.uniform(lo, hi, size=(4096, 3)).astype(np.float32)
        r = rng.uniform(*TUMOUR_RADIUS_MM, size=4096)
        inside = P["scene"].compute_occupancy(o3d.core.Tensor(c), nsamples=9).numpy() > 0.5
        dist = P["scene"].compute_distance(o3d.core.Tensor(c)).numpy()
        ok = inside & (dist >= r)
        centres += list(c[ok].astype(np.float64))
        radii += list(r[ok])
    return np.array(centres[:n]), np.array(radii[:n])


def spearman(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return float("nan")
    rx, ry = np.argsort(np.argsort(x)), np.argsort(np.argsort(y))
    return float(np.corrcoef(rx, ry)[0, 1])


# ==============================================================================
# Scoring one predicted pose
# ==============================================================================
def surface_dists(T_pred, P, vis_pts):
    """Distance (mm) of each visible point, taken back to the CT frame with T_pred, to the CT surface."""
    back = apply(np.linalg.inv(T_pred), vis_pts).astype(np.float32)
    return P["scene"].compute_distance(o3d.core.Tensor(back)).numpy()


def evaluate(T_pred, T_true, P, view, tumours):
    label, vis_pts, in_view = view
    liver, ignore = label == LIVER, (label == TIP) | (label == SHAFT)

    d = dice(render_mask(T_pred, P), liver, ignore)

    fit = float(surface_dists(T_pred, P, vis_pts).mean())

    c_true, c_pred = apply(T_true, tumours), apply(T_pred, tumours)
    err = np.linalg.norm(c_pred - c_true, axis=1)
    lateral = np.linalg.norm((c_pred - c_true)[:, :2], axis=1)  # across the view; depth = |dz|
    px = np.linalg.norm(project(c_pred, P["K"]) - project(c_true, P["K"]), axis=1)
    return d, fit, err, lateral, px


def run_icp(T_init, P, vis_pts):
    src = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(vis_pts))
    res = o3d.pipelines.registration.registration_icp(
        src, P["target"], 20.0, np.linalg.inv(T_init),
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=60))
    return np.linalg.inv(res.transformation)


# ==============================================================================
# Main
# ==============================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--patients", nargs="+", default=["01", "06", "10", "11"])
    ap.add_argument("--views", type=int, default=25, help="random test views per patient that pass the check")
    ap.add_argument("--perts", type=int, default=40, help="wrong predictions per family per view")
    ap.add_argument("--tumours", type=int, default=20, help="tumours per patient")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="benchmark/pilot_out")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rng = np.random.default_rng(args.seed)

    rows, per_tumour, excluded = [], [], []
    for pid in args.patients:
        P = load_patient(pid)
        tumours, radii = sample_tumours(P, rng, args.tumours)
        n_used = 0

        for idx in rng.permutation(P["views"]):
            if n_used == args.views:
                break
            label, vis_pts = load_view(P, int(idx))
            T_true = true_w2c(P["poses"][f"{idx:05d}"][0])

            # Sanity check: at the true pose the outline and the cloud must match the CT.
            # Views 2000-2499 of patients 10 and 11 fail it: an extra object hides part of the
            # liver, is labelled background, and its surface is included in the liver cloud.
            d0 = dice(render_mask(T_true, P), label == LIVER, (label == TIP) | (label == SHAFT))
            far0 = (surface_dists(T_true, P, vis_pts) > 3.0).mean()
            if d0 < 0.98 or far0 > 0.02:
                excluded.append((pid, int(idx), d0, far0))
                continue
            n_used += 1

            # A tumour is "in view" if its true centre projects onto the true liver outline
            gt_mask = render_mask(T_true, P)
            uv = project(apply(T_true, tumours), P["K"])
            ok = np.isfinite(uv).all(axis=1)
            ui, vi_ = np.full(len(uv), -1), np.full(len(uv), -1)
            ui[ok], vi_[ok] = np.round(uv[ok, 0]).astype(int), np.round(uv[ok, 1]).astype(int)
            inb = ok & (ui >= 0) & (ui < W) & (vi_ >= 0) & (vi_ < H)
            in_view = np.zeros(len(uv), bool)
            in_view[inb] = gt_mask[vi_[inb], ui[inb]]
            view = (label, vis_pts, in_view)

            print(f"[{pid}] view {idx:05d} ({n_used}/{args.views})  true pose: Dice {d0:.3f}, "
                  f"{100 * far0:.1f}% far points, {in_view.sum()}/{len(tumours)} tumours in view")

            pivots = {"camera": np.zeros(3), "visible": vis_pts.mean(axis=0),
                      "organ": apply(T_true, P["centroid"][None])[0]}
            for fam in FAMILIES:
                for _ in range(args.perts):
                    if fam in pivots:
                        T_pred = rigid_error(rng, pivots[fam]) @ T_true
                    elif fam == "scale":
                        s = np.exp(rng.uniform(*np.log(SCALE_RANGE)))
                        T_pred = np.diag([s, s, s, 1.0]) @ T_true
                    elif fam == "icp":
                        T_init = rigid_error(rng, pivots["organ"]) @ T_true
                        T_pred = run_icp(T_init, P, vis_pts)
                    else:
                        # The method sees a cloud with the wrong scale and aligns it rigidly
                        s = np.exp(rng.uniform(*np.log(SCALE_RECIPE_ERROR)))
                        T_init = np.diag([s, s, s, 1.0]) @ rigid_error(rng, pivots["organ"]) @ T_true
                        T_pred = run_icp(T_init, P, vis_pts * s)

                    seen = (label, vis_pts * s, in_view) if fam == "icp_scaled" else view
                    d, fit, err, lateral, px = evaluate(T_pred, T_true, P, seen, tumours)
                    E = T_pred @ np.linalg.inv(T_true)
                    scale = float(np.cbrt(np.linalg.det(E[:3, :3])))
                    rows.append(dict(
                        patient=pid, view=int(idx), family=fam, dice=d, surface_fit_mm=fit,
                        rot_err_deg=rotation_deg(E[:3, :3] / scale), scale=scale,
                        liver_centroid_err_mm=float(np.linalg.norm(apply(E, pivots["organ"][None])[0] - pivots["organ"])),
                        tumour_err_mean_mm=float(err.mean()), tumour_err_max_mm=float(err.max()),
                        tumour_lateral_mean_mm=float(lateral.mean()),
                        tumour_err_inview_mm=float(err[in_view].mean()) if in_view.any() else float("nan"),
                        tumour_px_inview=float(np.nanmean(px[in_view])) if in_view.any() else float("nan"),
                        n_inview=int(in_view.sum())))
                    per_tumour.append(np.stack([err, radii, in_view], axis=1))

    with open(os.path.join(args.out, "pilot_results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    np.save(os.path.join(args.out, "pilot_per_tumour.npy"), np.stack(per_tumour))
    with open(os.path.join(args.out, "excluded_views.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["patient", "view", "true_pose_dice", "far_point_fraction"])
        w.writerows(excluded)

    print(f"\n{len(excluded)} views drawn but excluded by the true-pose check "
          f"(see excluded_views.csv): " + ", ".join(
              f"{pid}: {sum(e[0] == pid for e in excluded)}" for pid in args.patients))
    summarize(rows, args.out)


def summarize(rows, out_dir):
    lines = []
    col = lambda key, sel: np.array([r[key] for r in sel], float)
    bins = [(0.0, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 0.95), (0.95, 1.01)]
    for fam in FAMILIES:
        sel = [r for r in rows if r["family"] == fam]
        dsc, err = col("dice", sel), col("tumour_err_mean_mm", sel)
        fit, inv = col("surface_fit_mm", sel), col("tumour_err_inview_mm", sel)
        lat = col("tumour_lateral_mean_mm", sel)
        lines.append(f"\n== {fam}  (n = {len(sel)}) ==")
        lines.append(f"Spearman rho  Dice vs tumour error: {spearman(dsc, err):+.2f}   "
                     f"surface fit vs tumour error: {spearman(fit, err):+.2f}")
        lines.append(f"{'Dice bin':<12}{'n':>6}{'median mm':>11}{'P90 mm':>9}{'> 10 mm':>9}"
                     f"{'in-view >10':>13}{'lateral >10':>13}")
        for lo, hi in bins:
            m = (dsc >= lo) & (dsc < hi)
            if not m.any():
                continue
            iv = inv[m][np.isfinite(inv[m])]
            lines.append(f"[{lo:.2f},{min(hi, 1):.2f}){m.sum():>6}{np.median(err[m]):>11.1f}"
                         f"{np.percentile(err[m], 90):>9.1f}{100 * (err[m] > MARGIN_MM).mean():>8.1f}%"
                         f"{(100 * (iv > MARGIN_MM).mean() if len(iv) else float('nan')):>12.1f}%"
                         f"{100 * (lat[m] > MARGIN_MM).mean():>12.1f}%")

    sel = [r for r in rows if r["family"] == PRIMARY]
    dsc, err = col("dice", sel), col("tumour_err_mean_mm", sel)
    good = dsc >= DICE_GOOD
    frac = (err[good] > MARGIN_MM).mean() if good.any() else float("nan")
    rho = spearman(dsc, err)
    if not good.any():
        verdict = f"INCONCLUSIVE (no {PRIMARY} prediction reached Dice >= {DICE_GOOD})"
    elif frac >= 0.10:
        verdict = "SUPPORTED"
    elif frac < 0.02 and abs(rho) >= 0.8:
        verdict = "NOT SUPPORTED"
    else:
        verdict = "INCONCLUSIVE"
    lines.append(f"\nPre-registered test ({PRIMARY}, Dice >= {DICE_GOOD}): n = {good.sum()}, "
                 f"{100 * frac:.1f}% with tumour error > {MARGIN_MM:.0f} mm, rho = {rho:+.2f} -> {verdict}")

    text = "\n".join(lines)
    print(text)
    with open(os.path.join(out_dir, "pilot_summary.txt"), "w") as f:
        f.write(text + "\n")
    plot(rows, out_dir)


def plot(rows, out_dir):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not installed; skipping the figure (pip install matplotlib)")
        return

    ink, muted, mark = "#2b2b2b", "#8a8a85", "#2a78d6"
    fams = [f for f in FAMILIES if f != "scale"]
    fig, axes = plt.subplots(2, len(fams), figsize=(4 * len(fams), 7), sharey=True)
    for j, fam in enumerate(fams):
        sel = [r for r in rows if r["family"] == fam]
        err = np.array([r["tumour_err_mean_mm"] for r in sel])
        for i, (key, xlabel) in enumerate([("dice", "Dice"), ("surface_fit_mm", "Surface fit (mm)")]):
            ax = axes[i, j]
            ax.scatter([r[key] for r in sel], err, s=6, alpha=0.35, color=mark, linewidths=0)
            ax.axhline(MARGIN_MM, color=muted, lw=1, ls="--")
            if key == "dice":
                ax.axvline(DICE_GOOD, color=muted, lw=1, ls="--")
            ax.set_xlabel(xlabel, color=ink)
            if j == 0:
                ax.set_ylabel("Tumour error (mm)", color=ink)
            if i == 0:
                ax.set_title(fam, color=ink)
            for s in ("top", "right"):
                ax.spines[s].set_visible(False)
            ax.tick_params(colors=muted)
    fig.suptitle("Dice and surface fit against tumour error (dashed: 10 mm margin, Dice 0.90)", color=ink)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "pilot_scatter.png"), dpi=130)
    print(f"figure saved to {os.path.join(out_dir, 'pilot_scatter.png')}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/registration/sim3_icp.py
"""
Module 3: fine alignment of the camera cloud to the CT surface, with scale.

Two-way ICP with scale. Each round pairs points in both directions, solves the similarity
transform for all pairs (Umeyama 1991), and repeats until the cloud stops moving:
  camera -> CT  every camera point should lie on the CT surface
  CT -> camera  every CT point the camera would see should have a camera point on it
With the first direction alone, a cloud that shrinks can always lie snugly somewhere, so the
scale collapses; the second direction penalises the CT surface a shrunk cloud leaves uncovered.
Optionally a third set of pairs uses the liver outline in the image:
  outline       the CT's outline seen from the camera should lie on the liver's outline
A smaller cloud with the camera moved closer satisfies both directions above, but makes the CT
spill over background pixels; the outline pairs pull it back.
The result maps camera points y to the CT frame as X = c Q y + d: the scale c corrects the
camera cloud about the camera centre, and (Q, d) is the camera-to-CT pose.
"""
import numpy as np
import open3d as o3d


def rotation(axis: np.ndarray, deg: float) -> np.ndarray:
    """Rotation by `deg` about the unit vector `axis` (Rodrigues)."""
    a = np.deg2rad(deg)
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(a) * K + (1 - np.cos(a)) * K @ K


def umeyama(src: np.ndarray, dst: np.ndarray):
    """Scale c, rotation Q and translation d minimising sum |dst - (c Q src + d)|^2."""
    mu_s, mu_d = src.mean(axis=0), dst.mean(axis=0)
    S, D = src - mu_s, dst - mu_d
    U, sig, Vt = np.linalg.svd(D.T @ S / len(src))
    E = np.diag([1.0, 1.0, np.sign(np.linalg.det(U @ Vt))])
    Q = U @ E @ Vt
    c = (sig * np.diag(E)).sum() / S.var(axis=0).sum()
    return c, Q, mu_d - c * Q @ mu_s


def visible_ct(scene, rays: np.ndarray, Q: np.ndarray, d: np.ndarray) -> np.ndarray:
    """CT points first hit by rays from the camera at pose (Q, d); rays are camera-frame directions.

    The scale does not move the camera centre, so it does not change the rays.
    """
    dirs = rays @ Q.T
    origins = np.broadcast_to(d, dirs.shape)
    t = scene.cast_rays(o3d.core.Tensor(np.hstack([origins, dirs]).astype(np.float32)))["t_hit"].numpy()
    hit = np.isfinite(t)
    return d + t[hit, None] * dirs[hit]


def nearest(points: np.ndarray, queries: np.ndarray) -> np.ndarray:
    """Index of the closest point to each query."""
    nns = o3d.core.nns.NearestNeighborSearch(o3d.core.Tensor(points.astype(np.float32)))
    nns.knn_index()
    return nns.knn_search(o3d.core.Tensor(queries.astype(np.float32)), 1)[0].numpy()[:, 0]


def closest(dist: np.ndarray, trim: float) -> np.ndarray:
    """Mask of the `trim` fraction of pairs with the smallest distance."""
    return dist <= np.quantile(dist, trim)


def edge(mask: np.ndarray) -> np.ndarray:
    """Pixels of the mask next to a pixel outside it; the image border does not count as outside."""
    p = np.pad(mask, 1, constant_values=True)
    return mask & ~(p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:])


def outline_pairs(scene, outline, c: float, Q: np.ndarray, d: np.ndarray, trim: float):
    """Pairs that pull the CT's outline, seen from the camera, onto the liver's outline in the image.

    outline: (camera-frame direction of every pixel of a coarse grid (h, w, 3), liver mask (h, w)).
    Each CT outline point is paired with the point at the same depth on the ray through the
    nearest pixel of the liver's outline.
    """
    dirs, mask = outline
    world = dirs.reshape(-1, 3) @ Q.T
    t = scene.cast_rays(o3d.core.Tensor(np.hstack([np.broadcast_to(d, world.shape), world])
                                        .astype(np.float32)))["t_hit"].numpy().reshape(mask.shape)
    ct_edge, liver_edge = edge(np.isfinite(t)), np.argwhere(edge(mask))
    if not ct_edge.any() or not len(liver_edge):
        return None
    X = d + t[ct_edge][:, None] * world.reshape(dirs.shape)[ct_edge]  # CT outline points, CT frame
    y = (X - d) @ Q / c                                                # the same, camera frame
    pix = lambda ij: np.hstack([ij, np.zeros((len(ij), 1))])           # noqa: E731
    target = liver_edge[nearest(pix(liver_edge), pix(np.argwhere(ct_edge)))]
    y_on = dirs[target[:, 0], target[:, 1]] * y[:, 2:3]                # on the liver-outline ray, same depth
    dist = c * np.linalg.norm(y_on - y, axis=1)
    keep = closest(dist, trim)
    return y_on[keep], X[keep], dist[keep]


def pairs(cloud: np.ndarray, rays: np.ndarray, scene, c: float, Q: np.ndarray, d: np.ndarray, trim: float,
          outline=None):
    """Matched (camera point, CT point) pairs in both directions and their distances (mm).

    cloud: camera points (mm); rays: camera-frame directions through liver pixels;
    scene: CT RaycastingScene (mm); outline: see `outline_pairs` (None: not used). Each set keeps
    its `trim` fraction of closest pairs, so stray points and tool-covered regions cannot drag the fit.
    """
    X = cloud @ (c * Q).T + d
    ct = scene.compute_closest_points(o3d.core.Tensor(X.astype(np.float32)))["points"].numpy()
    dist = np.linalg.norm(ct - X, axis=1)
    keep = closest(dist, trim)
    src, dst, err = [cloud[keep]], [ct[keep]], [dist[keep]]

    seen = visible_ct(scene, rays, Q, d)
    if len(seen):
        idx = nearest(X, seen)
        dist = np.linalg.norm(X[idx] - seen, axis=1)
        keep = closest(dist, trim)
        src.append(cloud[idx[keep]])
        dst.append(seen[keep])
        err.append(dist[keep])

    found = outline_pairs(scene, outline, c, Q, d, trim) if outline is not None else None
    if found is not None:
        src.append(found[0])
        dst.append(found[1])
        err.append(found[2])
    return np.vstack(src), np.vstack(dst), np.concatenate(err)


def fit_error(cloud: np.ndarray, rays: np.ndarray, scene, c: float, Q: np.ndarray, d: np.ndarray,
              trim: float = 0.9, outline=None) -> float:
    """Mean distance of the kept pairs in both directions: how well a pose explains the view.

    Measured in the camera cloud's own units (divided by the scale c): distances in the CT frame
    shrink with the cloud, so otherwise a shrunk cloud would score well just by being small.
    """
    return pairs(cloud, rays, scene, c, Q, d, trim, outline)[2].mean() / c


def register(cloud: np.ndarray, rays: np.ndarray, scene, c: float, Q: np.ndarray, d: np.ndarray,
             trim: float = 0.9, iters: int = 500, tol_mm: float = 1e-3, outline=None):
    """Refine the starting guess (c, Q, d); arguments as in `pairs`."""
    X = cloud @ (c * Q).T + d
    for _ in range(iters):
        src, dst, _ = pairs(cloud, rays, scene, c, Q, d, trim, outline)
        c, Q, d = umeyama(src, dst)
        X_new = cloud @ (c * Q).T + d
        if np.abs(X_new - X).max() < tol_mm:
            break
        X = X_new
    return c, Q, d

In [ ]:
%%writefile /kaggle/working/registration/capture_range.py
"""
Capture range of module 3 (sim3_icp.py) on clean synthetic training views.

Start from the true pose pushed off by a known amount, run module 3, and check whether it gets
back. Success: mean error at the patient's 200 probe tumours below 5 mm. Each error is tested
on its own (rotation, shift, size), then all three together.

The starting guess, applied to the true camera-to-CT map:
  size     the cloud is k times too big about the camera centre (as a wrong depth scale would be)
  rotation by `deg` about a random axis through the cloud centre
  shift    by `mm` in a random direction
Only training views are used (train_syn.txt); views whose cloud does not fit the CT at the true
pose (the extra object in patients 10 and 11) are skipped.

Usage: python registration/capture_range.py [--views-per-patient 20] [--tests size ...]
"""
import argparse
import csv
import os
import sys

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "benchmark"))
from pilot_dice_vs_tumour import LIVER, load_patient, load_view, sample_tumours, surface_dists, true_w2c  # noqa: E402
from sim3_icp import register, rotation  # noqa: E402

PATIENTS = ["01", "06", "10", "11"]
N_PROBES = 200
SUCCESS_MM = 5.0
TESTS = {  # name -> list of (rotation deg, shift mm, size factor)
    "rotation": [(deg, 0, 1.0) for deg in (5, 10, 20, 30, 45, 60, 90)],
    "shift": [(0, mm, 1.0) for mm in (5, 10, 20, 30, 50)],
    "size": [(0, 0, k) for k in (0.5, 0.6, 0.7, 0.8, 0.9, 1.1, 1.2, 1.3, 1.4, 1.6, 1.8, 2.0, 2.5)],
    "together": [(5, 5, 1.05), (10, 10, 1.1), (20, 15, 1.15), (30, 25, 1.2), (45, 35, 1.3)],
}


def unit(rng) -> np.ndarray:
    v = rng.normal(size=3)
    return v / np.linalg.norm(v)


def training_views(P, rng, n: int):
    """n random training views whose cloud fits the CT at the true pose, with rays through liver pixels.

    Every k-th liver pixel is used, so there are about as many rays as camera points and the two
    matching directions weigh equally.
    """
    with open(os.path.join(P["root"], "train_syn.txt")) as f:
        views = [int(line.strip().replace("ptsLiverCam", "")) for line in f if line.strip()]
    chosen = []
    for view in rng.permutation(views):
        label, cloud = load_view(P, view)
        T = true_w2c(P["poses"][f"{view:05d}"][0])
        if len(cloud) >= 100 and (surface_dists(T, P, cloud) > 3.0).mean() <= 0.02:
            rays = P["pix_dirs"][(label == LIVER).reshape(-1)]
            chosen.append((view, cloud, rays[::max(1, len(rays) // len(cloud))], T))
        if len(chosen) == n:
            return chosen
    return chosen


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--views-per-patient", type=int, default=20)
    ap.add_argument("--tests", nargs="+", choices=list(TESTS), default=list(TESTS))
    ap.add_argument("--out", default=os.path.join(HERE, "out"))
    args = ap.parse_args()
    tests = {name: TESTS[name] for name in args.tests}
    os.makedirs(args.out, exist_ok=True)

    rows = []
    for pid in PATIENTS:
        P = load_patient(pid)
        probes, _ = sample_tumours(P, np.random.default_rng([0, int(pid)]), N_PROBES)
        rng = np.random.default_rng([1, int(pid)])
        for view, cloud, rays, T in training_views(P, rng, args.views_per_patient):
            Q_true, d_true = T[:3, :3].T, -T[:3, :3].T @ T[:3, 3]  # camera -> CT
            m = Q_true @ cloud.mean(axis=0) + d_true                # cloud centre in the CT frame
            cam_true = (probes - d_true) @ Q_true                   # probes in the true camera frame
            for test, levels in tests.items():
                for deg, mm, k in levels:
                    if test == "together" and rng.random() < 0.5:
                        k = 1 / k                                   # too small as often as too big
                    dR = rotation(unit(rng), deg)
                    c, Q, d = register(cloud, rays, P["scene"], k, dR @ Q_true,
                                       dR @ (d_true - m) + m + mm * unit(rng))
                    tre = np.linalg.norm((probes - d) @ Q - cam_true, axis=1)
                    rows.append(dict(test=test, rotation_deg=deg, shift_mm=mm, size=round(k, 4),
                                     patient=pid, view=view, mean_tre_mm=tre.mean(),
                                     rotation_error_deg=np.rad2deg(np.arccos(np.clip(
                                         (np.trace(Q @ Q_true.T) - 1) / 2, -1, 1))),
                                     scale_error_pct=100 * (c - 1)))
            print(f"[{pid}] view {view:05d} done", flush=True)

    name = "capture_range" + ("" if len(tests) == len(TESTS) else "_" + "_".join(tests))
    with open(os.path.join(args.out, name + ".csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    for test, levels in tests.items():
        print(f"\n{test}: start error -> success (mean probe error < {SUCCESS_MM:.0f} mm), median error")
        for deg, mm, k in levels:
            sel = [r for r in rows if r["test"] == test and r["rotation_deg"] == deg and r["shift_mm"] == mm
                   and r["size"] in ((k, round(1 / k, 4)) if test == "together" else (k,))]
            tre = np.array([r["mean_tre_mm"] for r in sel])
            print(f"  {deg:>2} deg  {mm:>2} mm  x{k:<4}  {100 * (tre < SUCCESS_MM).mean():5.1f} %   "
                  f"{np.median(tre):6.1f} mm   ({len(sel)} runs)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/registration/multistart.py
"""
Module 2: rough pose and size from scratch, by trying many starting guesses.

Candidate cameras look at the CT liver from directions spread evenly over a sphere, with several
rolls about each direction. Each candidate puts the centre of the camera cloud on the first CT
surface point seen from that direction, with the cloud `start_size` times too big (module 3
recovers too-big starts but collapses from too-small ones). Every candidate gets a few rounds of
module 3, the `keep` best by two-way fit are refined fully, and the best fit wins.
"""
import numpy as np
import open3d as o3d

from sim3_icp import fit_error, register, rotation

START_SIZE = 1.25  # too small collapses suddenly, too big degrades gently (capture_range.py --tests size)
N_DIRS, N_ROLLS = 40, 8


def sphere_directions(n: int) -> np.ndarray:
    """n unit vectors spread evenly over the sphere (Fibonacci lattice)."""
    i = np.arange(n) + 0.5
    z = 1 - 2 * i / n
    phi = np.pi * (1 + 5 ** 0.5) * i
    r = np.sqrt(1 - z ** 2)
    return np.stack([r * np.cos(phi), r * np.sin(phi), z], axis=1)


def candidates(cloud: np.ndarray, scene, centre: np.ndarray, start_size: float, n_dirs: int, n_rolls: int):
    """Starting guesses (c, Q, d), each with the cloud centre on the CT surface facing one direction."""
    m = cloud.mean(axis=0)
    u = m / np.linalg.norm(m)  # direction from the camera to the cloud centre, camera frame
    starts = []
    for v in sphere_directions(n_dirs):  # viewing direction in the CT frame
        far = centre - 1000.0 * v
        t = scene.cast_rays(o3d.core.Tensor(np.hstack([far, v])[None].astype(np.float32)))["t_hit"].numpy()[0]
        if not np.isfinite(t):
            continue
        axis = np.cross(u, v)
        align = rotation(axis / (np.linalg.norm(axis) + 1e-12), np.rad2deg(np.arccos(np.clip(u @ v, -1, 1))))
        for roll in np.arange(n_rolls) * 360.0 / n_rolls:
            Q = rotation(v, roll) @ align  # sends u to v, then spins about v
            starts.append((start_size, Q, far + t * v - start_size * Q @ m))
    return starts


def register_from_scratch(cloud: np.ndarray, rays: np.ndarray, scene, centre: np.ndarray,
                          start_size: float = START_SIZE, n_dirs: int = N_DIRS, n_rolls: int = N_ROLLS,
                          screen_iters: int = 20, keep: int = 10, outline=None):
    """Best pose (c, Q, d), plus all fully refined candidates and their fit errors (mm).

    outline: optional liver outline in the image, passed to module 3 (see sim3_icp.outline_pairs).
    """
    screened = [register(cloud, rays, scene, *s, iters=screen_iters, outline=outline)
                for s in candidates(cloud, scene, centre, start_size, n_dirs, n_rolls)]
    best = sorted(screened, key=lambda p: fit_error(cloud, rays, scene, *p, outline=outline))[:keep]
    finals = [register(cloud, rays, scene, *p, outline=outline) for p in best]
    errors = [fit_error(cloud, rays, scene, *p, outline=outline) for p in finals]
    return finals[int(np.argmin(errors))], finals, errors

In [ ]:
%%writefile /kaggle/working/registration/outline_reg.py
"""
Rigid registration of the CT liver to one image, from the liver outline and, optionally, the ridge.

No point cloud: the CT is in mm and the camera is calibrated, so where the outline is, its shape
and its size fix the whole pose, depth included. Each round pairs points in the image (pixels)
both ways and takes one Gauss-Newton step on the pose (R, t: CT -> camera, mm):
  outline  CT outline seen from the camera  <->  liver outline in the mask
  ridge    CT ridge seen from the camera    <->  ridge drawn in the image
Each set keeps its 90 % closest pairs and counts as a whole (its mean), the ridge times
`ridge_weight`. Pixels next to a tool are ignored: the liver's edge there is unknown.
Search from scratch tries 40 viewing directions x 8 rolls (as module 2), each starting at the
distance where the CT looks as big as the mask.
"""
import numpy as np
import open3d as o3d

from multistart import N_DIRS, N_ROLLS, sphere_directions
from sim3_icp import closest, edge, nearest, rotation

STEP = 4      # coarse pixel grid for rendering the CT outline
TRIM = 0.9


def grow(mask: np.ndarray) -> np.ndarray:
    p = np.pad(mask, 1)
    return mask | p[:-2, 1:-1] | p[2:, 1:-1] | p[1:-1, :-2] | p[1:-1, 2:]


def make_view(K: np.ndarray, liver: np.ndarray, tool: np.ndarray, ridge2d=None) -> dict:
    """One image: K, liver and tool masks (full size), ridge pixels (n x 2, optional)."""
    H, W = liver.shape
    v, u = np.mgrid[0:H:STEP, 0:W:STEP]
    pix = np.stack([u + 0.5, v + 0.5], axis=-1)
    dirs = np.dstack([(pix[..., 0] - K[0, 2]) / K[0, 0], (pix[..., 1] - K[1, 2]) / K[1, 1], np.ones(u.shape)])
    unknown = grow(tool[::STEP, ::STEP])
    return dict(K=K, size=(W, H), pix=pix, dirs=dirs, unknown=unknown, liver=liver[::STEP, ::STEP],
                liver_edge=pix[edge(liver[::STEP, ::STEP]) & ~unknown], ridge2d=ridge2d)


def project(K: np.ndarray, Xc: np.ndarray) -> np.ndarray:
    return np.stack([K[0, 0] * Xc[:, 0] / Xc[:, 2] + K[0, 2], K[1, 1] * Xc[:, 1] / Xc[:, 2] + K[1, 2]], axis=1)


def render(scene, view, R, t) -> np.ndarray:
    """Ray length to the CT per grid pixel (inf where the ray misses)."""
    world = view["dirs"].reshape(-1, 3) @ R
    origins = np.broadcast_to(-R.T @ t, world.shape)
    return scene.cast_rays(o3d.core.Tensor(np.hstack([origins, world]).astype(np.float32)))["t_hit"] \
        .numpy().reshape(view["dirs"].shape[:2])


def ct_outline(scene, view, R, t):
    """CT outline seen from the camera: CT points (CT frame) and their pixels."""
    hit = render(scene, view, R, t)
    keep = edge(np.isfinite(hit)) & ~view["unknown"]
    return (view["dirs"][keep] * hit[keep, None] - t) @ R, view["pix"][keep]


def visible(scene, view, R, t, X: np.ndarray) -> np.ndarray:
    """Mask of CT points X seen by the camera: in front, inside the image, not hidden, not under a tool."""
    Xc = X @ R.T + t
    p = project(view["K"], np.where(Xc[:, 2:] > 1e-6, Xc, np.nan))
    W, H = view["size"]
    ok = (Xc[:, 2] > 1e-6) & (p[:, 0] >= 0) & (p[:, 0] < W) & (p[:, 1] >= 0) & (p[:, 1] < H)
    i, j = (np.nan_to_num(p[:, 1]) // STEP).astype(int), (np.nan_to_num(p[:, 0]) // STEP).astype(int)
    ok &= ~view["unknown"][np.clip(i, 0, view["unknown"].shape[0] - 1), np.clip(j, 0, view["unknown"].shape[1] - 1)]
    C = -R.T @ t
    dist = np.linalg.norm(X - C, axis=1)
    rays = np.hstack([np.broadcast_to(C, X.shape), (X - C) / dist[:, None]]).astype(np.float32)
    return ok & (scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy() >= dist - 1.0)


def match(src_px: np.ndarray, dst_px: np.ndarray) -> np.ndarray:
    """Index into dst of the nearest pixel to each src pixel."""
    z = lambda p: np.hstack([p, np.zeros((len(p), 1))])  # noqa: E731
    return nearest(z(dst_px), z(src_px))


def pairs(scene, view, R, t, ridge3d=None, ridge_weight=1.0):
    """Pair sets as (CT points, target pixels, weight of the set)."""
    out = []
    X, px = ct_outline(scene, view, R, t)
    m = view["liver_edge"]
    if len(X) and len(m):
        out.append((X, m[match(px, m)], 0.5))            # CT outline -> liver outline
        out.append((X[match(m, px)], m, 0.5))            # liver outline -> CT outline
    r = view["ridge2d"]
    if ridge3d is not None and r is not None and len(r):
        seen = ridge3d[visible(scene, view, R, t, ridge3d)]
        front = ridge3d[(ridge3d @ R.T + t)[:, 2] > 1e-6]
        if len(seen):
            out.append((seen, r[match(project(view["K"], seen @ R.T + t), r)], 0.5 * ridge_weight))
        if len(front):
            out.append((front[match(r, project(view["K"], front @ R.T + t))], r, 0.5 * ridge_weight))
    kept = []
    for X, target, w in out:
        d = np.linalg.norm(project(view["K"], X @ R.T + t) - target, axis=1)
        k = closest(d, TRIM)
        kept.append((X[k], target[k], w, d[k]))
    return kept


def fit_error(scene, view, R, t, ridge3d=None, ridge_weight=1.0) -> float:
    """Weighted mean pixel distance of the kept pairs (inf if the CT outline is not in view)."""
    kept = pairs(scene, view, R, t, ridge3d, ridge_weight)
    if len(kept) < 2:
        return np.inf
    return sum(w * d.mean() for _, _, w, d in kept)


def gn_step(K, R, t, kept, rotate=True):
    """One Gauss-Newton step: rotation w about the camera centre and shift dt, both in the camera frame."""
    A, b = np.zeros((6, 6)), np.zeros(6)
    for X, target, w, _ in kept:
        Xc = X @ R.T + t
        x, y, z = Xc.T
        Jp = np.zeros((len(X), 2, 3))
        Jp[:, 0, 0], Jp[:, 0, 2] = K[0, 0] / z, -K[0, 0] * x / z ** 2
        Jp[:, 1, 1], Jp[:, 1, 2] = K[1, 1] / z, -K[1, 1] * y / z ** 2
        S = np.zeros((len(X), 3, 3))  # d(Xc) = w x Xc + dt = -[Xc]x w + dt
        S[:, 0, 1], S[:, 0, 2], S[:, 1, 0], S[:, 1, 2], S[:, 2, 0], S[:, 2, 1] = z, -y, -z, x, y, -x
        J = np.concatenate([Jp @ S, Jp], axis=2)
        r = project(K, Xc) - target
        A += w / len(X) * np.einsum("nij,nik->jk", J, J)
        b += w / len(X) * np.einsum("nij,ni->j", J, r)
    free = slice(0, 6) if rotate else slice(3, 6)
    A, b = A[free, free], b[free]
    step = np.zeros(6)
    step[free] = np.linalg.solve(A + 1e-6 * np.trace(A) * np.eye(len(b)), -b)
    angle = np.linalg.norm(step[:3])
    dR = rotation(step[:3] / angle, np.rad2deg(angle)) if angle > 0 else np.eye(3)
    return dR @ R, dR @ t + step[3:]


def register(scene, view, R, t, ridge3d=None, ridge_weight=1.0, iters=100, rotate=True, tol=1e-3):
    """Refine the pose (R, t)."""
    for _ in range(iters):
        kept = pairs(scene, view, R, t, ridge3d, ridge_weight)
        if len(kept) < 2:
            break
        R_new, t_new = gn_step(view["K"], R, t, kept, rotate)
        moved = np.linalg.norm(R_new - R) * 100 + np.linalg.norm(t_new - t)
        R, t = R_new, t_new
        if moved < tol:
            break
    return R, t


def starts(scene, view, centre: np.ndarray, n_dirs: int = N_DIRS, n_rolls: int = N_ROLLS):
    """Poses looking at the CT centre along the ray through the mask's centre, at matching size."""
    m = view["pix"][view["liver"]].mean(axis=0)
    K = view["K"]
    u = np.array([(m[0] - K[0, 2]) / K[0, 0], (m[1] - K[1, 2]) / K[1, 1], 1.0])
    u /= np.linalg.norm(u)
    out = []
    for v in sphere_directions(n_dirs):  # viewing direction in the CT frame
        axis = np.cross(u, v)
        align = rotation(axis / (np.linalg.norm(axis) + 1e-12), np.rad2deg(np.arccos(np.clip(u @ v, -1, 1))))
        for roll in np.arange(n_rolls) * 360.0 / n_rolls:
            R = (rotation(v, roll) @ align).T  # CT -> camera; sends v to u
            area = np.isfinite(render(scene, view, R, -R @ (centre - 300.0 * v))).sum()
            if area:
                dist = 300.0 * np.sqrt(area / view["liver"].sum())
                out.append((R, -R @ (centre - dist * v)))
    return out


def register_from_scratch(scene, view, centre, ridge3d=None, ridge_weight=1.0, n_dirs=N_DIRS, n_rolls=N_ROLLS,
                          screen_iters=15, keep=10):
    """Best pose (R, t) and its fit error."""
    err = lambda p: fit_error(scene, view, *p, ridge3d, ridge_weight)  # noqa: E731
    screened = [register(scene, view, *s, ridge3d, ridge_weight, iters=screen_iters)
                for s in starts(scene, view, centre, n_dirs, n_rolls)]
    finals = [register(scene, view, *p, ridge3d, ridge_weight) for p in sorted(screened, key=err)[:keep]]
    errors = [err(p) for p in finals]
    return finals[int(np.argmin(errors))], min(errors)

In [ ]:
%%writefile /kaggle/working/registration/ct_landmarks.py
"""
3D ridge of a CT liver, found from its shape alone (no cameras, no scanner axes).

The liver has two surfaces: the smooth dome under the diaphragm and the grooved underside that faces
the gut. They meet at a rim that is thin and sharp at the front (the inferior border) and rounded at
the back. The ridge is the sharp part of that rim:
  1. fold   how sharply the surface bends within 8 mm of each vertex (+ convex edge, - groove)
  2. split  a minimum cut between dome seeds (outer shell, facing away from the underside) and
            underside seeds (the deepest pits, and surface facing the same way as them); cutting is
            cheap where the surface folds sharply, so the cut follows the rim
  3. ridge  rim vertices that fold more than 95 % of the dome does, extended along the rim while they
            fold more than 80 % of it; pieces shorter than 3 cm are dropped

Frozen on 2026-11-10, before the one-time check against Koo et al.'s expert ridge on the LLR-LUS
livers. Do not tune it on LLR-LUS.

Saved to registration/landmarks/<patient>.json as {"ridge": [[vertex indices of one piece, in order], ...]}
on the CT mesh as loaded by benchmark/pilot_dice_vs_tumour.py (mm).

Usage: python registration/ct_landmarks.py [--patients 01 06 10 11] [--show]
"""
import argparse
import json
import os

import numpy as np
import open3d as o3d
import scipy.sparse as sp
from scipy.sparse.csgraph import breadth_first_order, connected_components, dijkstra, maximum_flow

from capture_range import HERE
from pilot_dice_vs_tumour import DATA_DIR

FOLD_MM = 8.0
START, KEEP = 95, 80   # percentiles of the dome's fold
MIN_PIECE_MM = 30.0


def load_mesh(pid: str) -> o3d.geometry.TriangleMesh:
    mesh = o3d.io.read_triangle_mesh(os.path.join(DATA_DIR, pid, "model", "reconstructed_mesh_world_m.obj"))
    mesh.scale(1000.0, center=[0, 0, 0])
    mesh.compute_vertex_normals()
    return mesh


def fold(V, N):
    """Signed fold per vertex: spread of the normals within FOLD_MM (0 flat .. 1), + convex, - concave."""
    tree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(V)))
    f = np.zeros(len(V))
    for i in range(len(V)):
        nb = np.asarray(tree.search_radius_vector_3d(V[i], FOLD_MM)[1])
        f[i] = (1 - np.linalg.norm(N[nb].mean(0))) * np.sign(np.sum((V[nb] - V[i]) * (N[nb] - N[i])))
    return f


def hull_depth(mesh):
    """How far each vertex lies inside the convex hull (mm): ~0 on the outer shell, large in pits."""
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh.compute_convex_hull()[0]))
    return scene.compute_distance(o3d.core.Tensor(np.asarray(mesh.vertices, np.float32))).numpy()


def graph(V, F):
    """Symmetric vertex graph of the mesh, weighted by edge length (mm)."""
    e = np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]])
    G = sp.coo_matrix((np.linalg.norm(V[e[:, 0]] - V[e[:, 1]], axis=1), (e[:, 0], e[:, 1])), shape=(len(V),) * 2).tocsr()
    return G.maximum(G.T)


def split(V, N, F, G, f, depth):
    """True for dome vertices: minimum cut between dome and underside seeds, cheap across sharp convex folds."""
    i, j = np.unique(np.sort(np.vstack([F[:, [0, 1]], F[:, [1, 2]], F[:, [2, 0]]]), 1), axis=0).T  # every edge once
    sharp = np.clip(f[[i, j]].mean(0), 0, None) / np.percentile(np.abs(f), 98)
    cap = np.maximum(1, np.round(100 * np.linalg.norm(V[i] - V[j], axis=1) * np.exp(-5 * sharp))).astype(np.int32)
    pits = depth > np.percentile(depth, 90)   # porta hepatis, gallbladder bed, fissures: always underside
    u = V[pits].mean(0) - V.mean(0)
    u /= np.linalg.norm(u)                    # the way the underside faces
    dome_seed = np.flatnonzero((depth < 1.0) & (N @ u < -0.5))
    under_seed = np.flatnonzero(pits | (N @ u > 0.5))
    n, big = len(V), np.int32(1 << 30)
    rows = np.r_[i, j, np.full(len(dome_seed), n), under_seed]
    cols = np.r_[j, i, dome_seed, np.full(len(under_seed), n + 1)]
    vals = np.r_[cap, cap, np.full(len(dome_seed) + len(under_seed), big)].astype(np.int32)
    C = sp.csr_matrix((vals, (rows, cols)), shape=(n + 2, n + 2))
    res = C - maximum_flow(C, n, n + 1).flow
    res.data[res.data < 0] = 0
    res.eliminate_zeros()
    dome = np.zeros(n + 2, bool)
    dome[breadth_first_order(res, n, return_predecessors=False)] = True
    dome = dome[:n]
    for side in (True, False):  # each surface in one piece: small islands (e.g. a pit in the dome) join the other side
        m = dome == side
        _, lab = connected_components(G[m][:, m], directed=False)
        dome[np.flatnonzero(m)[lab != np.bincount(lab).argmax()]] = not side
    return dome


def longest_path(G, nodes):
    """Vertices of the longest shortest path through `nodes` (assumed connected), in order."""
    Gb = G[nodes][:, nodes]
    d = dijkstra(Gb, indices=0)
    a = int(np.argmax(np.where(np.isfinite(d), d, -1)))
    d, pred = dijkstra(Gb, indices=a, return_predecessors=True)
    b = int(np.argmax(np.where(np.isfinite(d), d, -1)))
    path = [b]
    while path[-1] != a:
        path.append(pred[path[-1]])
    return nodes[path]


def ridge(mesh) -> list:
    """Ridge pieces of a liver mesh in mm (with vertex normals), each an ordered array of vertex indices."""
    V, N, F = np.asarray(mesh.vertices), np.asarray(mesh.vertex_normals), np.asarray(mesh.triangles)
    G = graph(V, F)
    f = fold(V, N)
    dome = split(V, N, F, G, f, hull_depth(mesh))
    i, j = G.nonzero()
    rim = np.unique(i[dome[i] & ~dome[j]])   # dome side of the cut
    weak = rim[f[rim] > np.percentile(f[dome], KEEP)]
    strong = rim[f[rim] > np.percentile(f[dome], START)]
    _, lab = connected_components(G[weak][:, weak], directed=False)
    sharp = weak[np.isin(lab, lab[np.isin(weak, strong)])]
    _, lab = connected_components(G[sharp][:, sharp], directed=False)
    pieces = [longest_path(G, sharp[lab == c]) for c in range(lab.max() + 1)]
    return [p for p in pieces if np.linalg.norm(np.diff(V[p], axis=0), axis=1).sum() >= MIN_PIECE_MM]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--patients", nargs="+", default=["01", "06", "10", "11"])
    ap.add_argument("--show", action="store_true", help="open a 3D window per patient (grey liver, red ridge)")
    args = ap.parse_args()

    os.makedirs(os.path.join(HERE, "landmarks"), exist_ok=True)
    for pid in args.patients:
        mesh = load_mesh(pid)
        pieces = ridge(mesh)
        V = np.asarray(mesh.vertices)
        json.dump({"ridge": [p.tolist() for p in pieces]}, open(os.path.join(HERE, "landmarks", f"{pid}.json"), "w"))
        print(f"{pid}: ridge pieces (mm) " + ", ".join(f"{np.linalg.norm(np.diff(V[p], axis=0), axis=1).sum():.0f}" for p in pieces))
        if args.show:
            colors = np.full((len(V), 3), 0.75)
            tree = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(mesh.vertices))
            for i in np.concatenate(pieces):
                colors[tree.search_radius_vector_3d(V[i], 2.5)[1]] = [0.9, 0.1, 0.1]
            mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
            o3d.visualization.draw_geometries([mesh], window_name=f"patient {pid}: ridge (red)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/registration/test_ridge_value.py
"""
Does a ridge help mask-only registration beyond searching harder, and what ambiguity does the outline leave?

On clean synthetic training views (true pose known), outline_reg.py registers the CT from scratch:
  outline       liver mask only; 40 viewing directions x 8 rolls, 10 candidates refined fully
  ridge         the same + a perfect ridge: the CT ridge (ct_landmarks.py, frozen) seen at the true
                pose (in view, not hidden, not under a tool), projected into the image, weight 1
  outline_more  liver mask only, a 4x bigger search: 80 x 16 starts, 40 candidates refined
"Perfect" means a detector that marks exactly our 3D rule's ridge; it measures how much a right
ridge could add, not how good our rule or a detector is. outline_more is the control: on
patients 01, 06, 10 and 11 most failures were search misses, which more search might fix as well.

Scored by the mean error at the patient's 200 probe tumours (as capture_range.py); success < 10 mm
(the LLR-LUS inclusion margin), < 5 mm also reported.

Each failure (>= 10 mm) is classified with T*, the pose refined from the true pose:
  search       fit(F) > fit(T*) + 0.5 px: a better-fitting pose exists, the search missed it
  alternative  otherwise: the outline fits the wrong pose F as well as the right one; then the
               path from T* to F (rotation and camera centre interpolated, 9 steps, shift
               re-fitted at each) decides
                 separate   fit rises > 1 px above both ends somewhere on the path: a distinct wrong pose
                 connected  otherwise: a smooth valley of poses that fit equally
T* itself drifting >= 10 mm from the truth is a connected valley at the truth.
Views whose mask has < 20 outline pixels on the coarse grid are skipped (no outline to fit).

Run 1 (2026-11-10, patients 01 06 10 11, 20 views each, outline and ridge only) was exploratory:
the design was built and looked at on those patients. Run 2 (all P2I-LReg patients, 10 views each,
all three conditions; fixed on 2026-11-11 before it started) is confirmatory:
  primary   success(ridge) - success(outline_more) on the 17 patients other than 01 06 10 11,
            paired per view; 95 % CI from 2000 bootstrap resamples of patients (seed 0)
            CI above 0: the ridge adds beyond search | below 0: the ridge hurts | otherwise: inconclusive
  secondary the same for ridge - outline and outline_more - outline; failure kinds per condition;
            all of it again for patients 01 06 10 11, reported separately

Usage: python registration/test_ridge_value.py [--data-root dataset/liver-registration] [--patients 01 06 ...]
           [--views-per-patient 10] [--workers 4] [--out registration/out/ridge_value.csv] [--summary-only]
The CSV is written row by row; a rerun skips the (patient, view, condition) rows already in it.
"""
import argparse
import csv
import json
import os
import multiprocessing

import numpy as np

import pilot_dice_vs_tumour as pilot
from capture_range import HERE, N_PROBES, training_views
from ct_landmarks import ridge
from outline_reg import fit_error, make_view, project, register, register_from_scratch, visible
from pilot_dice_vs_tumour import LIVER, SHAFT, TIP, load_patient, load_view, rotation_deg, sample_tumours
from sim3_icp import rotation

CONDITIONS = {  # name -> (ridge used, viewing directions, rolls, candidates refined fully)
    "outline": (False, 40, 8, 10),
    "ridge": (True, 40, 8, 10),
    "outline_more": (False, 80, 16, 40),
}
DEV_PATIENTS = ["01", "06", "10", "11"]
SUCCESS_MM = 10.0
ALT_PX, BARRIER_PX, PATH_STEPS = 0.5, 1.0, 9
FIELDS = ["patient", "view", "condition", "ridge_points_seen", "error_mm", "rotation_error_deg", "fit_px",
          "truth_refined_error_mm", "truth_refined_fit_px", "failure", "barrier_px"]
_patients = {}


def path_barrier(scene, view, A, B, ridge3d):
    """Highest fit on the way from pose A to pose B, above the higher end (px)."""
    (Ra, ta), (Rb, tb) = A, B
    Ca, Cb = -Ra.T @ ta, -Rb.T @ tb
    rel = Ra @ Rb.T
    angle = rotation_deg(rel)
    w = np.array([rel[2, 1] - rel[1, 2], rel[0, 2] - rel[2, 0], rel[1, 0] - rel[0, 1]])
    axis = w / (np.linalg.norm(w) + 1e-12)
    worst = 0.0
    for k in np.arange(1, PATH_STEPS + 1) / (PATH_STEPS + 1):
        R = rotation(axis, -k * angle) @ Ra  # Ra -> Rb
        R, t = register(scene, view, R, -R @ ((1 - k) * Ca + k * Cb), ridge3d, iters=20, rotate=False)
        worst = max(worst, fit_error(scene, view, R, t, ridge3d))
    return worst - max(fit_error(scene, view, *A, ridge3d), fit_error(scene, view, *B, ridge3d))


def ridge_points(P, pid):
    """The patient's CT ridge vertices (mm); computed with the frozen rule and saved if not there yet."""
    path = os.path.join(HERE, "landmarks", f"{pid}.json")
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        json.dump({"ridge": [p.tolist() for p in ridge(P["mesh"])]}, open(path, "w"))
    return np.asarray(P["mesh"].vertices)[np.concatenate(json.load(open(path))["ridge"])]


def patient(data_root, pid):
    if pid not in _patients:
        pilot.DATA_DIR = data_root
        P = load_patient(pid)
        P["ridge3d"] = ridge_points(P, pid)
        P["probes"] = sample_tumours(P, np.random.default_rng([0, int(pid)]), N_PROBES)[0]
        _patients.clear()  # one patient in memory per worker
        _patients[pid] = P
    return _patients[pid]


def run_view(task):
    """All missing conditions for one view: list of CSV rows."""
    data_root, pid, view_id, T, conditions = task
    P = patient(data_root, pid)
    label, _ = load_view(P, view_id)
    R0, t0 = T[:3, :3], T[:3, 3]
    view = make_view(P["K"], label == LIVER, (label == TIP) | (label == SHAFT))
    if len(view["liver_edge"]) < 20:
        return []
    seen = P["ridge3d"][visible(P["scene"], view, R0, t0, P["ridge3d"])]
    view["ridge2d"] = project(P["K"], seen @ R0.T + t0)
    probes = P["probes"]
    err = lambda R, t: np.linalg.norm(probes @ R.T + t - (probes @ R0.T + t0), axis=1).mean()  # noqa: E731

    rows = []
    for cond in conditions:
        use_ridge, n_dirs, n_rolls, keep = CONDITIONS[cond]
        rid = P["ridge3d"] if use_ridge else None
        T_star = register(P["scene"], view, R0, t0, rid)
        F, fit_F = register_from_scratch(P["scene"], view, P["centroid"], rid, n_dirs=n_dirs, n_rolls=n_rolls, keep=keep)
        fit_star = fit_error(P["scene"], view, *T_star, rid)
        row = dict(patient=pid, view=view_id, condition=cond, ridge_points_seen=len(seen), error_mm=err(*F),
                   rotation_error_deg=rotation_deg(F[0] @ R0.T), fit_px=fit_F, truth_refined_error_mm=err(*T_star),
                   truth_refined_fit_px=fit_star, failure="", barrier_px="")
        if row["error_mm"] >= SUCCESS_MM:
            if fit_F > fit_star + ALT_PX:
                row["failure"] = "search"
            else:
                row["barrier_px"] = path_barrier(P["scene"], view, T_star, F, rid)
                row["failure"] = "separate" if row["barrier_px"] > BARRIER_PX else "connected"
        rows.append(row)
    return rows


def bootstrap_ci(diff_by_patient, n=2000, seed=0):
    """95 % CI of the mean per-view difference, resampling patients."""
    groups = list(diff_by_patient.values())
    rng = np.random.default_rng(seed)
    means = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])


def summary(rows):
    by = {}
    for r in rows:
        by.setdefault((r["patient"], int(r["view"])), {})[r["condition"]] = r
    for name, group in (("17 new patients (confirmatory)", lambda p: p not in DEV_PATIENTS),
                        ("patients 01 06 10 11 (development)", lambda p: p in DEV_PATIENTS)):
        views = {k: v for k, v in by.items() if group(k[0]) and len(v) == len(CONDITIONS)}
        if not views:
            continue
        print(f"\n{name}: {len(views)} views, {len({k[0] for k in views})} patients")
        for cond in CONDITIONS:
            e = np.array([float(v[cond]["error_mm"]) for v in views.values()])
            kinds = {k: sum(v[cond]["failure"] == k for v in views.values()) for k in ("search", "separate", "connected")}
            drift = sum(float(v[cond]["truth_refined_error_mm"]) >= SUCCESS_MM for v in views.values())
            print(f"  {cond:12s} < {SUCCESS_MM:.0f} mm {100 * (e < SUCCESS_MM).mean():5.1f} %  < 5 mm {100 * (e < 5).mean():5.1f} %  "
                  f"median {np.median(e):5.1f} mm  failures {kinds}  truth drifts: {drift}")
        for a, b in (("ridge", "outline_more"), ("ridge", "outline"), ("outline_more", "outline")):
            diff = {}
            for (pid, _), v in views.items():
                ok = lambda c: float(v[c]["error_mm"]) < SUCCESS_MM  # noqa: E731
                diff.setdefault(pid, []).append(float(ok(a)) - float(ok(b)))
            diff = {p: np.array(d) for p, d in diff.items()}
            mean = np.concatenate(list(diff.values())).mean()
            lo, hi = bootstrap_ci(diff)
            fixed = sum((d > 0).sum() for d in diff.values())
            broke = sum((d < 0).sum() for d in diff.values())
            verdict = "adds" if lo > 0 else "hurts" if hi < 0 else "inconclusive"
            print(f"  {a} - {b}: {100 * mean:+.1f} points (95 % CI {100 * lo:+.1f} to {100 * hi:+.1f}), "
                  f"{fixed} views fixed, {broke} broken -> {verdict}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default=pilot.DATA_DIR)
    ap.add_argument("--patients", nargs="+", help="default: every patient folder in --data-root")
    ap.add_argument("--views-per-patient", type=int, default=10)
    ap.add_argument("--workers", type=int, default=1)
    ap.add_argument("--out", default=os.path.join(HERE, "out", "ridge_value.csv"))
    ap.add_argument("--summary-only", action="store_true")
    args = ap.parse_args()
    patients = args.patients or sorted(p for p in os.listdir(args.data_root) if p.isdigit()
                                       and os.path.exists(os.path.join(args.data_root, p, "syn", "camPose.yml")))

    rows = list(csv.DictReader(open(args.out))) if os.path.exists(args.out) else []
    if not args.summary_only:
        done = {(r["patient"], int(r["view"]), r["condition"]) for r in rows}
        tasks = []
        for pid in patients:
            P = patient(args.data_root, pid)  # also computes the ridge once, before the workers start
            for view_id, _, _, T in training_views(P, np.random.default_rng([1, int(pid)]), args.views_per_patient):
                todo = [c for c in CONDITIONS if (pid, view_id, c) not in done]
                if todo:
                    tasks.append((args.data_root, pid, view_id, T, todo))
        print(f"{len(tasks)} views to run on {len(patients)} patients", flush=True)
        os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
        new_file = not os.path.exists(args.out)
        _patients.clear()  # workers start fresh ("spawn"): forking a process with Open3D's threads running can hang
        with open(args.out, "a", newline="") as f, multiprocessing.get_context("spawn").Pool(args.workers) as pool:
            w = csv.DictWriter(f, fieldnames=FIELDS)
            if new_file:
                w.writeheader()
            for k, out in enumerate(pool.imap_unordered(run_view, tasks), 1):
                w.writerows(out)
                f.flush()
                rows += [{key: str(val) for key, val in r.items()} for r in out]
                print(f"[{k}/{len(tasks)}] " + "  ".join(f"{r['patient']}/{r['view']:05d} {r['condition']} "
                                                          f"{r['error_mm']:.1f} mm {r['failure']}" for r in out), flush=True)
    summary(rows)


if __name__ == "__main__":
    main()

## 4. Finished views from earlier versions

In [ ]:
import shutil
OUT = "/kaggle/working/ridge_out"
CSV = f"{OUT}/ridge_value.csv"
os.makedirs(OUT, exist_ok=True)
os.makedirs("/kaggle/working/registration/landmarks", exist_ok=True)
for prev in glob.glob("/kaggle/input/" + "**/ridge_out/ridge_value.csv", recursive=True):
    if not os.path.exists(CSV):
        shutil.copy2(prev, CSV)
        print("copied", prev)
for prev in glob.glob("/kaggle/input/" + "**/registration/landmarks/*.json", recursive=True):
    dst = os.path.join("/kaggle/working/registration/landmarks", os.path.basename(prev))
    if not os.path.exists(dst):
        shutil.copy2(prev, dst)
print(sum(1 for _ in open(CSV)) - 1 if os.path.exists(CSV) else 0, "rows already done")

## 5. Run
The ridge of each CT is computed once with the frozen rule (`ct_landmarks.py`) and saved in `registration/landmarks/`.

In [ ]:
%cd /kaggle/working
cmd = (f"PYTHONPATH=registration:benchmark timeout {HOURS}h python -W ignore registration/test_ridge_value.py "
       f"--data-root {ROOT} --patients {' '.join(ORDER)} --views-per-patient {VIEWS_PER_PATIENT} "
       f"--workers {WORKERS} --out {CSV}")
!{cmd}; echo "exit code $? (124: stopped after {HOURS} h, run a new version to finish)"

## 6. Summary (finished views only)

In [ ]:
!PYTHONPATH=registration:benchmark python -W ignore registration/test_ridge_value.py --data-root {ROOT} --out {CSV} --summary-only

## 7. Download

In [ ]:
!cd /kaggle/working && zip -qr ridge_value_results.zip ridge_out registration && ls -la ridge_value_results.zip